# Deepfake Audio Detection (v4 - class weighting + threshold tuning)
**Run on Kaggle with GPU T4 enabled.**

Changes vs v3:
- Class-weighted loss to fix deepfake class bias
- Threshold tuned to 0.4 to catch more deepfakes
- Optimal threshold search added
- Everything else same as v3

In [ ]:
!pip install -q librosa soundfile torchaudio scikit-learn tqdm seaborn

In [ ]:
import os, glob, random, json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Locate dataset and build file lists

In [ ]:
BASE = '/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm'
TRAIN_DIR = os.path.join(BASE, 'training')
VAL_DIR   = os.path.join(BASE, 'validation')
TEST_DIR  = os.path.join(BASE, 'testing')

def collect_files(data_dir):
    rows = []
    for sub in os.listdir(data_dir):
        sub_path = os.path.join(data_dir, sub)
        if not os.path.isdir(sub_path):
            continue
        sub_l = sub.lower()
        if sub_l == 'real':   label = 0
        elif sub_l == 'fake': label = 1
        else: continue
        for f in glob.glob(os.path.join(sub_path, '*.wav')):
            rows.append((f, label))
    return pd.DataFrame(rows, columns=['path', 'label'])

train_df = collect_files(TRAIN_DIR)
val_df   = collect_files(VAL_DIR)
test_df  = collect_files(TEST_DIR)

print('Train:', train_df.shape)
print(train_df['label'].value_counts())
print('Val:', val_df.shape)
print(val_df['label'].value_counts())
print('Test:', test_df.shape)
print(test_df['label'].value_counts())

assert len(train_df) > 0 and len(val_df) > 0, 'No files found — check BASE path'

## 2. Feature extraction + SpecAugment

In [ ]:
SR       = 16000
DURATION = 4.0
N_SAMPLES = int(SR * DURATION)
N_MELS   = 64

def load_audio(path):
    y, sr = librosa.load(path, sr=SR, mono=True)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    else:
        y = y[:N_SAMPLES]
    return y

def to_logmel(y):
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, n_fft=1024, hop_length=256)
    logmel = librosa.power_to_db(mel, ref=np.max)
    logmel = (logmel - logmel.min()) / (logmel.max() - logmel.min() + 1e-6)
    return logmel.astype(np.float32)

def spec_augment(mel, freq_mask=8, time_mask=20, n_freq_masks=2, n_time_masks=2):
    mel = mel.copy()
    n_mels, n_time = mel.shape
    for _ in range(n_freq_masks):
        if n_mels - freq_mask > 0:
            f0 = np.random.randint(0, n_mels - freq_mask)
            mel[f0:f0+freq_mask, :] = 0
    for _ in range(n_time_masks):
        if n_time - time_mask > 0:
            t0 = np.random.randint(0, n_time - time_mask)
            mel[:, t0:t0+time_mask] = 0
    return mel

class AudioDataset(Dataset):
    def __init__(self, df, augment=False):
        self.paths   = df['path'].values
        self.labels  = df['label'].values
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            y   = load_audio(self.paths[idx])
            mel = to_logmel(y)
        except Exception:
            mel = np.zeros((N_MELS, int(N_SAMPLES/256)+1), dtype=np.float32)
        if self.augment:
            mel = spec_augment(mel)
        mel   = torch.from_numpy(mel).unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

train_ds = AudioDataset(train_df, augment=True)
val_ds   = AudioDataset(val_df,   augment=False)
test_ds  = AudioDataset(test_df,  augment=False)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

sample_mel, sample_label = train_ds[0]
print('Mel shape:', sample_mel.shape, 'Label:', sample_label)

## 3. Model

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,  16, 3, padding=1), nn.BatchNorm2d(16),  nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1)),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

model = AudioCNN().to(DEVICE)
print(model)

## 4. Training (class-weighted loss + early stopping)

In [ ]:
EPOCHS         = 30
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
PATIENCE       = 5
LABEL_SMOOTHING = 0.05

# Class-weighted loss: penalise whichever class has fewer samples more
class_counts  = train_df['label'].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum()
class_weights = class_weights.to(DEVICE)
print('Class weights:', class_weights)

criterion  = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train: optimizer.zero_grad()
            out  = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds = out.argmax(1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_labels, all_preds

best_val_loss    = float('inf')
best_val_acc     = 0
epochs_no_improve = 0
history          = []

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    print(f'Epoch {epoch}/{EPOCHS} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | '
          f'val_loss={val_loss:.4f} acc={val_acc:.4f} | lr={optimizer.param_groups[0]["lr"]:.2e}')

    if val_loss < best_val_loss - 1e-4:
        best_val_loss     = val_loss
        best_val_acc      = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print('  -> saved best model')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'  -> early stopping triggered')
            break

print('Best val_loss:', best_val_loss, '| Best val_acc:', best_val_acc)

In [ ]:
import matplotlib.pyplot as plt
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train_loss')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='val_loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('epoch')
axes[1].plot(hist_df['epoch'], hist_df['train_acc'], label='train_acc')
axes[1].plot(hist_df['epoch'], hist_df['val_acc'],   label='val_acc')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('epoch')
plt.savefig('training_curves.png', bbox_inches='tight')
plt.show()

## 5. Find optimal threshold on validation set

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Collect val probs
val_probs, val_labels = [], []
with torch.no_grad():
    for x, y in tqdm(val_loader, leave=False):
        x = x.to(DEVICE)
        out = model(x)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        val_probs.extend(probs)
        val_labels.extend(y.numpy())

val_probs  = np.array(val_probs)
val_labels = np.array(val_labels)

# Search best threshold by F1 on val set
best_thresh = 0.5
best_f1     = 0
for thresh in np.arange(0.2, 0.8, 0.01):
    preds = (val_probs >= thresh).astype(int)
    f1    = f1_score(val_labels, preds, zero_division=0)
    acc   = accuracy_score(val_labels, preds)
    if f1 > best_f1 and acc >= 0.80:   # must also meet accuracy target
        best_f1     = f1
        best_thresh = thresh

print(f'Best threshold: {best_thresh:.2f} | Val F1: {best_f1:.4f}')

## 6. Evaluate on test set

In [ ]:
all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in tqdm(test_loader, leave=False):
        x = x.to(DEVICE)
        out   = model(x)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds  = (all_probs >= best_thresh).astype(int)

print('Prediction distribution:', np.bincount(all_preds, minlength=2))

acc           = accuracy_score(all_labels, all_preds)
f1            = f1_score(all_labels, all_preds)
cm            = confusion_matrix(all_labels, all_preds)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fpr, tpr, _ = roc_curve(all_labels, all_probs)
fnr         = 1 - tpr
eer_idx     = np.nanargmin(np.absolute(fnr - fpr))
eer         = (fpr[eer_idx] + fnr[eer_idx]) / 2

print(f'Threshold used : {best_thresh:.2f}')
print(f'Accuracy       : {acc:.4f}')
print(f'F1 Score       : {f1:.4f}')
print(f'EER            : {eer*100:.2f}%')
print('Confusion Matrix (rows=true, cols=pred) [0=Genuine, 1=Deepfake]:')
print(cm)
print(f'Per-class acc  : Genuine={per_class_acc[0]:.4f}, Deepfake={per_class_acc[1]:.4f}')

In [ ]:
import seaborn as sns

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Genuine','Deepfake'],
            yticklabels=['Genuine','Deepfake'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

## 7. Save report

In [ ]:
report = {
    'threshold'        : float(best_thresh),
    'accuracy'         : float(acc),
    'f1_score'         : float(f1),
    'eer_percent'      : float(eer*100),
    'confusion_matrix' : cm.tolist(),
    'per_class_accuracy': {
        'Genuine' : float(per_class_acc[0]),
        'Deepfake': float(per_class_acc[1])
    },
    'best_val_loss': float(best_val_loss),
    'best_val_acc' : float(best_val_acc),
    'epochs_trained': len(history)
}
with open('performance_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print('\nSaved: best_model.pth, performance_report.json, confusion_matrix.png, training_curves.png')

## 8. Inference function

In [ ]:
INFERENCE_THRESHOLD = best_thresh   # saved from val search above

def predict_audio(path, model=model, device=DEVICE, threshold=INFERENCE_THRESHOLD):
    model.eval()
    y   = load_audio(path)
    mel = to_logmel(y)
    x   = torch.from_numpy(mel).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        out   = model(x)
        probs = torch.softmax(out, dim=1)[0]
    prob_deepfake = float(probs[1])
    pred          = 1 if prob_deepfake >= threshold else 0
    label         = 'Deepfake (AI-Generated)' if pred == 1 else 'Genuine (Human)'
    confidence    = prob_deepfake if pred == 1 else float(probs[0])
    return label, confidence

# Quick test
sample_path = test_df.iloc[0]['path']
print(predict_audio(sample_path))
print('True label:', 'Deepfake' if test_df.iloc[0]['label']==1 else 'Genuine')